In [1]:
# ============================================================
# DEEPSHIELD-AI — VIDEO EVALUATION
# STEP 1: IMPORTS & CONFIGURATION
# ============================================================

import os
import cv2
import numpy as np

import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 65)
print("DEEPSHIELD-AI — VIDEO EVALUATION")
print("=" * 65)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NUM_FRAMES = 16
FRAME_SIZE = 224
NUM_CLASSES = 2

CNN_FEATURES = 512
LSTM_HIDDEN_SIZE = 256

print("Device:", DEVICE)

DEEPSHIELD-AI — VIDEO EVALUATION
Device: cuda


In [2]:
# ============================================================
# STEP 2 — BEST MODEL PATH
# ============================================================

BEST_MODEL_PATH = os.path.join(
    "models",
    "video",
    "video_resnet18_lstm_best.pth"
)

print("Model path:")
print(BEST_MODEL_PATH)

print("\nModel exists:", os.path.isfile(BEST_MODEL_PATH))

Model path:
models\video\video_resnet18_lstm_best.pth

Model exists: True


In [3]:
# ============================================================
# STEP 3 — VIDEO MODEL ARCHITECTURE
# ============================================================

class VideoResNet18LSTM(nn.Module):

    def __init__(
        self,
        cnn,
        cnn_features=512,
        hidden_size=256,
        num_layers=1,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = cnn

        self.lstm = nn.LSTM(
            input_size=cnn_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        batch_size, num_frames, channels, height, width = x.shape

        x = x.reshape(
            batch_size * num_frames,
            channels,
            height,
            width
        )

        features = self.cnn(x)

        features = features.reshape(
            batch_size,
            num_frames,
            -1
        )

        lstm_output, _ = self.lstm(features)

        final_feature = lstm_output[:, -1, :]

        final_feature = self.dropout(
            final_feature
        )

        output = self.classifier(
            final_feature
        )

        return output


print("VideoResNet18LSTM architecture recreated.")

VideoResNet18LSTM architecture recreated.


In [4]:
# ============================================================
# STEP 4 — LOAD BEST VIDEO MODEL
# ============================================================

from torchvision import models

resnet = models.resnet18(
    weights=None
)

resnet.fc = nn.Identity()

video_model = VideoResNet18LSTM(
    cnn=resnet,
    cnn_features=CNN_FEATURES,
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=1,
    num_classes=NUM_CLASSES,
    dropout=0.3
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

video_model.load_state_dict(
    checkpoint["model_state_dict"]
)

video_model = video_model.to(DEVICE)
video_model.eval()

print("=" * 65)
print("BEST MODEL LOADED")
print("=" * 65)

print("Checkpoint epoch:",
      checkpoint["epoch"])

print("Validation accuracy:",
      checkpoint["best_val_accuracy"])

print("Device:",
      next(video_model.parameters()).device)

BEST MODEL LOADED
Checkpoint epoch: 5
Validation accuracy: 0.375
Device: cuda:0


In [5]:
# ============================================================
# STEP 5 — EVALUATION TRANSFORM
# ============================================================

evaluation_transform = transforms.Compose([
    transforms.Resize(
        (FRAME_SIZE, FRAME_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("=" * 65)
print("EVALUATION TRANSFORM READY")
print("=" * 65)

print("Frame size:", FRAME_SIZE)
print("Frames/video:", NUM_FRAMES)
print("Output tensor: [3, 224, 224]")

EVALUATION TRANSFORM READY
Frame size: 224
Frames/video: 16
Output tensor: [3, 224, 224]


In [6]:
# ============================================================
# STEP 6 — LOAD SDFVD DATASET
# ============================================================

from datasets import load_dataset

SEED = 42

sdfvd_dataset = load_dataset(
    "Hemgg/SDFVD-video-dataset",
    split="train"
)

print("=" * 65)
print("SDFVD DATASET LOADED")
print("=" * 65)

print("Total videos :", len(sdfvd_dataset))
print("Features     :", sdfvd_dataset.features)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/106 [00:00<?, ?it/s]

SDFVD DATASET LOADED
Total videos : 106
Features     : {'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'), 'label': ClassLabel(names=['Fake', 'Real'])}


In [7]:
# ============================================================
# STEP 7 — BUILD VIDEO METADATA
# ============================================================

video_metadata = []

for i in range(len(sdfvd_dataset)):

    raw_video = (
        sdfvd_dataset.data
        .column("video")[i]
        .as_py()
    )

    video_path = raw_video["path"]
    label = int(sdfvd_dataset["label"][i])

    video_metadata.append({
        "index": i,
        "video_path": video_path,
        "label": label
    })

print("=" * 65)
print("VIDEO METADATA CREATED")
print("=" * 65)

print("Total:", len(video_metadata))

print("\nFirst sample:")
print(video_metadata[0])

VIDEO METADATA CREATED
Total: 106

First sample:
{'index': 0, 'video_path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4', 'label': 0}


In [8]:
# ============================================================
# STEP 8 — RECREATE ORIGINAL SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

metadata_indices = np.arange(
    len(video_metadata)
)

metadata_labels = np.array([
    item["label"]
    for item in video_metadata
])

train_indices, temp_indices = train_test_split(
    metadata_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=metadata_labels
)

temp_labels = metadata_labels[temp_indices]

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("=" * 65)
print("ORIGINAL SPLIT RECREATED")
print("=" * 65)

print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))
print("Total      :",
      len(train_indices)
      + len(val_indices)
      + len(test_indices))

ORIGINAL SPLIT RECREATED
Train      : 74
Validation : 16
Test       : 16
Total      : 106


In [9]:
# ============================================================
# STEP 9 — TEST SET CLASS CHECK
# ============================================================

test_labels = [
    video_metadata[i]["label"]
    for i in test_indices
]

fake_count = test_labels.count(0)
real_count = test_labels.count(1)

print("=" * 65)
print("TEST SET CLASS DISTRIBUTION")
print("=" * 65)

print("FAKE :", fake_count)
print("REAL :", real_count)
print("Total:", len(test_labels))

TEST SET CLASS DISTRIBUTION
FAKE : 8
REAL : 8
Total: 16


In [10]:
# ============================================================
# STEP 10 — EVALUATION VIDEO DATASET
# ============================================================

from torch.utils.data import Dataset

class EvaluationVideoDataset(Dataset):

    def __init__(
        self,
        metadata,
        indices,
        num_frames=NUM_FRAMES,
        transform=None
    ):
        self.metadata = metadata
        self.indices = list(indices)
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        metadata_index = self.indices[idx]

        item = self.metadata[metadata_index]

        video_path = item["video_path"]
        label = int(item["label"])

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video: {video_path}"
            )

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if total_frames <= 0:
            cap.release()
            raise RuntimeError(
                f"Invalid video: {video_path}"
            )

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            self.num_frames,
            dtype=int
        )

        frames = []

        for frame_index in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_index)
            )

            success, frame = cap.read()

            if not success:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            image = Image.fromarray(frame)

            if self.transform is not None:
                image = self.transform(image)

            frames.append(image)

        cap.release()

        if len(frames) == 0:
            raise RuntimeError(
                f"No frames extracted: {video_path}"
            )

        while len(frames) < self.num_frames:
            frames.append(
                frames[-1].clone()
            )

        frames = frames[:self.num_frames]

        video_tensor = torch.stack(frames)

        return video_tensor, label


print("EvaluationVideoDataset created successfully.")

EvaluationVideoDataset created successfully.


In [12]:
# ============================================================
# STEP 11 — CREATE TEST DATASET
# ============================================================

test_dataset = EvaluationVideoDataset(
    metadata=video_metadata,
    indices=test_indices,
    num_frames=NUM_FRAMES,
    transform=evaluation_transform
)

print("=" * 65)
print("TEST DATASET")
print("=" * 65)

print("Test videos:", len(test_dataset))

TEST DATASET
Test videos: 16


In [14]:
# ============================================================
# STEP 12 — CREATE TEST DATALOADER
# ============================================================

from torch.utils.data import DataLoader

TEST_BATCH_SIZE = 2

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("=" * 65)
print("TEST DATALOADER")
print("=" * 65)

print("Test videos :", len(test_dataset))
print("Batch size  :", TEST_BATCH_SIZE)
print("Test batches:", len(test_loader))

TEST DATALOADER
Test videos : 16
Batch size  : 2
Test batches: 8


In [15]:
# ============================================================
# STEP 13 — TEST BATCH CHECK
# ============================================================

test_videos, test_labels = next(
    iter(test_loader)
)

print("=" * 65)
print("TEST BATCH CHECK")
print("=" * 65)

print("Video shape :", test_videos.shape)
print("Label shape :", test_labels.shape)
print("Labels      :", test_labels)
print("Data type   :", test_videos.dtype)

TEST BATCH CHECK
Video shape : torch.Size([2, 16, 3, 224, 224])
Label shape : torch.Size([2])
Labels      : tensor([0, 0])
Data type   : torch.float32


In [16]:
# ============================================================
# STEP 14 — GPU BATCH CHECK
# ============================================================

test_videos = test_videos.to(DEVICE)
test_labels = test_labels.to(DEVICE)

print("=" * 65)
print("GPU BATCH CHECK")
print("=" * 65)

print("Video device:", test_videos.device)
print("Label device:", test_labels.device)

GPU BATCH CHECK
Video device: cuda:0
Label device: cuda:0


In [17]:
# ============================================================
# STEP 15 — SINGLE BATCH PREDICTION
# ============================================================

video_model.eval()

with torch.no_grad():

    outputs = video_model(
        test_videos
    )

    probabilities = torch.softmax(
        outputs,
        dim=1
    )

    predictions = torch.argmax(
        probabilities,
        dim=1
    )

print("=" * 65)
print("SINGLE BATCH PREDICTION")
print("=" * 65)

print("Outputs shape:")
print(outputs.shape)

print("\nProbabilities:")
print(probabilities)

print("\nPredictions:")
print(predictions)

print("\nTrue labels:")
print(test_labels)

SINGLE BATCH PREDICTION
Outputs shape:
torch.Size([2, 2])

Probabilities:
tensor([[0.2882, 0.7118],
        [0.7973, 0.2027]], device='cuda:0')

Predictions:
tensor([1, 0], device='cuda:0')

True labels:
tensor([0, 0], device='cuda:0')


In [18]:
# ============================================================
# STEP 16 — COMPLETE TEST INFERENCE
# ============================================================

all_labels = []
all_predictions = []
all_probabilities = []

video_model.eval()

with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(DEVICE)

        outputs = video_model(
            videos
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

print("=" * 65)
print("COMPLETE TEST INFERENCE")
print("=" * 65)

print("Total test samples :", len(all_labels))
print("Predictions shape  :", all_predictions.shape)
print("Probabilities shape:", all_probabilities.shape)

COMPLETE TEST INFERENCE
Total test samples : 16
Predictions shape  : (16,)
Probabilities shape: (16, 2)


In [19]:
# ============================================================
# STEP 17 — TEST ACCURACY
# ============================================================

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("=" * 65)
print("TEST ACCURACY")
print("=" * 65)

print(
    f"Accuracy : {test_accuracy:.4f}"
)

print(
    f"Accuracy : {test_accuracy * 100:.2f}%"
)

TEST ACCURACY
Accuracy : 0.3125
Accuracy : 31.25%


In [20]:
# ============================================================
# STEP 18 — TEST CLASSIFICATION METRICS
# ============================================================

test_precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

test_recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

test_f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

print("=" * 65)
print("TEST CLASSIFICATION METRICS")
print("=" * 65)

print(
    f"Precision : {test_precision:.4f}"
)

print(
    f"Recall    : {test_recall:.4f}"
)

print(
    f"F1 Score  : {test_f1:.4f}"
)

TEST CLASSIFICATION METRICS
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000


In [21]:
# ============================================================
# STEP 18A — PREDICTION DISTRIBUTION CHECK
# ============================================================

print("=" * 65)
print("PREDICTION DISTRIBUTION CHECK")
print("=" * 65)

print("\nTrue labels:")
print(all_labels)

print("\nPredictions:")
print(all_predictions)

print("\nTrue label counts:")
print("FAKE (0):", np.sum(all_labels == 0))
print("REAL (1):", np.sum(all_labels == 1))

print("\nPredicted label counts:")
print("FAKE (0):", np.sum(all_predictions == 0))
print("REAL (1):", np.sum(all_predictions == 1))

PREDICTION DISTRIBUTION CHECK

True labels:
[0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 1]

Predictions:
[1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]

True label counts:
FAKE (0): 8
REAL (1): 8

Predicted label counts:
FAKE (0): 13
REAL (1): 3


In [22]:
# ============================================================
# STEP 19 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1]
)

print("=" * 65)
print("TEST CONFUSION MATRIX")
print("=" * 65)

print("              Predicted")
print("             FAKE  REAL")
print(
    f"Actual FAKE   {cm[0,0]:>3}   {cm[0,1]:>3}"
)
print(
    f"Actual REAL   {cm[1,0]:>3}   {cm[1,1]:>3}"
)

TEST CONFUSION MATRIX
              Predicted
             FAKE  REAL
Actual FAKE     5     3
Actual REAL     8     0


In [23]:
# ============================================================
# STEP 20 — CLASSIFICATION REPORT
# ============================================================

print("=" * 65)
print("DEEPSHIELD-AI — VIDEO TEST REPORT")
print("=" * 65)

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1],
        target_names=["FAKE", "REAL"],
        zero_division=0
    )
)

DEEPSHIELD-AI — VIDEO TEST REPORT
              precision    recall  f1-score   support

        FAKE       0.38      0.62      0.48         8
        REAL       0.00      0.00      0.00         8

    accuracy                           0.31        16
   macro avg       0.19      0.31      0.24        16
weighted avg       0.19      0.31      0.24        16



In [24]:
# ============================================================
# STEP 21 — VIDEO PREDICTION CONFIDENCE ANALYSIS
# ============================================================

fake_probabilities = all_probabilities[:, 0]
real_probabilities = all_probabilities[:, 1]

print("=" * 65)
print("VIDEO PREDICTION CONFIDENCE")
print("=" * 65)

for i in range(len(all_labels)):

    true_label = (
        "FAKE"
        if all_labels[i] == 0
        else "REAL"
    )

    predicted_label = (
        "FAKE"
        if all_predictions[i] == 0
        else "REAL"
    )

    print(
        f"Video {i:02d} | "
        f"True: {true_label:<5} | "
        f"Pred: {predicted_label:<5} | "
        f"FAKE: {fake_probabilities[i]:.4f} | "
        f"REAL: {real_probabilities[i]:.4f}"
    )

VIDEO PREDICTION CONFIDENCE
Video 00 | True: FAKE  | Pred: REAL  | FAKE: 0.2882 | REAL: 0.7118
Video 01 | True: FAKE  | Pred: FAKE  | FAKE: 0.7973 | REAL: 0.2027
Video 02 | True: FAKE  | Pred: FAKE  | FAKE: 0.8348 | REAL: 0.1652
Video 03 | True: REAL  | Pred: FAKE  | FAKE: 0.8392 | REAL: 0.1608
Video 04 | True: REAL  | Pred: FAKE  | FAKE: 0.8392 | REAL: 0.1608
Video 05 | True: FAKE  | Pred: FAKE  | FAKE: 0.8119 | REAL: 0.1881
Video 06 | True: FAKE  | Pred: FAKE  | FAKE: 0.6402 | REAL: 0.3598
Video 07 | True: REAL  | Pred: FAKE  | FAKE: 0.8865 | REAL: 0.1135
Video 08 | True: REAL  | Pred: FAKE  | FAKE: 0.7916 | REAL: 0.2084
Video 09 | True: REAL  | Pred: FAKE  | FAKE: 0.9312 | REAL: 0.0688
Video 10 | True: REAL  | Pred: FAKE  | FAKE: 0.6538 | REAL: 0.3462
Video 11 | True: REAL  | Pred: FAKE  | FAKE: 0.8485 | REAL: 0.1515
Video 12 | True: FAKE  | Pred: REAL  | FAKE: 0.1709 | REAL: 0.8291
Video 13 | True: FAKE  | Pred: FAKE  | FAKE: 0.7716 | REAL: 0.2284
Video 14 | True: FAKE  | Pred: REA

In [25]:
# ============================================================
# STEP 22 — PER-VIDEO EVALUATION RESULTS
# ============================================================

import pandas as pd

evaluation_results = []

for i in range(len(all_labels)):

    true_label = (
        "FAKE"
        if all_labels[i] == 0
        else "REAL"
    )

    predicted_label = (
        "FAKE"
        if all_predictions[i] == 0
        else "REAL"
    )

    fake_confidence = float(
        fake_probabilities[i]
    )

    real_confidence = float(
        real_probabilities[i]
    )

    evaluation_results.append({
        "Video": i,
        "True Label": true_label,
        "Predicted Label": predicted_label,
        "Fake Confidence": fake_confidence,
        "Real Confidence": real_confidence,
        "Correct": (
            true_label == predicted_label
        )
    })

evaluation_df = pd.DataFrame(
    evaluation_results
)

print("=" * 65)
print("DEEPSHIELD-AI — VIDEO EVALUATION TABLE")
print("=" * 65)

display(evaluation_df)

DEEPSHIELD-AI — VIDEO EVALUATION TABLE


,Video,True Label,Predicted Label,Fake Confidence,Real Confidence,Correct
0,0,FAKE,REAL,0.288235,0.711765,False
1,1,FAKE,FAKE,0.797277,0.202723,True
2,2,FAKE,FAKE,0.834761,0.165239,True
3,3,REAL,FAKE,0.839202,0.160798,False
4,4,REAL,FAKE,0.839182,0.160818,False
5,5,FAKE,FAKE,0.811869,0.188131,True
6,6,FAKE,FAKE,0.640229,0.359771,True
7,7,REAL,FAKE,0.886521,0.113479,False
8,8,REAL,FAKE,0.791596,0.208404,False
9,9,REAL,FAKE,0.931196,0.068804,False


In [26]:
# ============================================================
# STEP 23 — DEEPSHIELD VIDEO RISK SCORE
# ============================================================

evaluation_df["Risk Score"] = (
    evaluation_df["Fake Confidence"] * 100
)

def risk_level(score):

    if score >= 75:
        return "HIGH RISK"

    elif score >= 50:
        return "MEDIUM RISK"

    else:
        return "LOW RISK"


evaluation_df["Risk Level"] = (
    evaluation_df["Risk Score"]
    .apply(risk_level)
)

print("=" * 65)
print("DEEPSHIELD-AI — VIDEO RISK ANALYSIS")
print("=" * 65)

display(
    evaluation_df[
        [
            "Video",
            "True Label",
            "Predicted Label",
            "Fake Confidence",
            "Risk Score",
            "Risk Level",
            "Correct"
        ]
    ]
)

DEEPSHIELD-AI — VIDEO RISK ANALYSIS


,Video,True Label,Predicted Label,Fake Confidence,Risk Score,Risk Level,Correct
0,0,FAKE,REAL,0.288235,28.823456,LOW RISK,False
1,1,FAKE,FAKE,0.797277,79.727721,HIGH RISK,True
2,2,FAKE,FAKE,0.834761,83.476079,HIGH RISK,True
3,3,REAL,FAKE,0.839202,83.920163,HIGH RISK,False
4,4,REAL,FAKE,0.839182,83.918238,HIGH RISK,False
5,5,FAKE,FAKE,0.811869,81.186938,HIGH RISK,True
6,6,FAKE,FAKE,0.640229,64.022857,MEDIUM RISK,True
7,7,REAL,FAKE,0.886521,88.652098,HIGH RISK,False
8,8,REAL,FAKE,0.791596,79.159576,HIGH RISK,False
9,9,REAL,FAKE,0.931196,93.119645,HIGH RISK,False


In [27]:
# ============================================================
# STEP 24 — RISK STATISTICS
# ============================================================

print("=" * 65)
print("VIDEO RISK STATISTICS")
print("=" * 65)

print(
    f"Average Fake Confidence : "
    f"{fake_probabilities.mean():.4f}"
)

print(
    f"Average Real Confidence : "
    f"{real_probabilities.mean():.4f}"
)

print(
    f"Average Risk Score      : "
    f"{evaluation_df['Risk Score'].mean():.2f}"
)

print("\nRisk Levels:")

print(
    evaluation_df["Risk Level"]
    .value_counts()
)


VIDEO RISK STATISTICS
Average Fake Confidence : 0.6827
Average Real Confidence : 0.3173
Average Risk Score      : 68.27

Risk Levels:
Risk Level
HIGH RISK      11
LOW RISK        3
MEDIUM RISK     2
Name: count, dtype: int64


In [28]:
# ============================================================
# STEP 25 — SAVE VIDEO EVALUATION RESULTS
# ============================================================

EVALUATION_DIR = os.path.join(
    "models",
    "video"
)

os.makedirs(
    EVALUATION_DIR,
    exist_ok=True
)

RESULTS_PATH = os.path.join(
    EVALUATION_DIR,
    "video_test_results.csv"
)

evaluation_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("=" * 65)
print("EVALUATION RESULTS SAVED")
print("=" * 65)

print("Saved at:")
print(RESULTS_PATH)

print(
    "File exists:",
    os.path.isfile(RESULTS_PATH)
)

EVALUATION RESULTS SAVED
Saved at:
models\video\video_test_results.csv
File exists: True
